# Jem assessment — Stage 1: analytical dataset

Run cells in order. Upload the seven original CSVs when prompted in Colab.
This stage produces auditable historical Wednesday snapshots and a snapshot for
every registered employee in the latest reporting week. It does not fit a model,
impute durations, classify notes, or generate breach predictions.

**Hours convention.** A shift belongs wholly to its `shift_date`, including an
overnight end the following morning. This reproduces the supplied weekly export.
Times are assumed to be local South African wall-clock times; elapsed duration is
treated as worked hours because break information is absent. Shifts are assumed
shorter than 24 hours. Equal start/end times are ambiguous and flagged, not
interpreted as a 24-hour shift.

**Availability assumption.** A reporting day includes completed overnight records
associated with that date. This is a shift-date snapshot, NOT a literal midnight
snapshot. Twelve current Wednesday records finish on Thursday in the supplied
export. There are no record-creation/revision timestamps, so point-in-time
availability cannot be proven. Historical replay assumes the records existed in
their supplied form by the reporting cut-off under this convention.

**Scope.** Use the assessment rule `weekly_hours > 55`. Payment premiums do not
multiply hours. Client-requested hours still count. Retain the latest represented
week for current scoring; never use it as historical ground truth in this stage.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:  # Also allows these functions to run in a normal Python script.
    display = print

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 70)

BREACH_HOURS = 55.0
HISTORICAL_CUTOFF_DAY = 2  # Monday=0; Wednesday=2. Change to 1 or 3 later.
DATA_DIR = Path("/content") if Path("/content").exists() else Path("project_sources")
FILENAMES = ["employees.csv", "sites.csv", "shifts.csv", "shift_notes.csv",
             "public_holidays.csv", "weekly_summary.csv", "payroll_details.csv"]

def locate_file(name):
    # Accept original names and the numbered copies supplied for this analysis.
    matches = list(DATA_DIR.glob(name)) + list(DATA_DIR.glob(f"*-{name}"))
    assert len(matches) <= 1, f"Multiple copies of {name}; select one export."
    return matches[0] if matches else None

missing_files = [name for name in FILENAMES if locate_file(name) is None]
if missing_files:
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"Place {missing_files} in {DATA_DIR.resolve()}")
    print("Upload the missing CSV files:", missing_files)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for name, content in files.upload().items():
        (DATA_DIR / Path(name).name).write_bytes(content)

assert all(locate_file(name) is not None for name in FILENAMES), "Missing input files."
# Read strings first: preserve IDs, leading zeros and literal note text such as n/a.
raw = {name[:-4]: pd.read_csv(locate_file(name), dtype="string", keep_default_na=False)
       for name in FILENAMES}
employees = raw["employees"].copy()
sites = raw["sites"].copy()
notes = raw["shift_notes"].copy()
holidays = raw["public_holidays"].copy()
summary = raw["weekly_summary"].copy()
payroll = raw["payroll_details"].copy()


## Input inspection and integrity

The inspection lists every file's shape and fields without displaying payroll
banking or tax data. Payroll is loaded and its employee keys checked; it contributes
no features. Note text is preserved unchanged. Blank notes are different from the
literal placeholder `n/a`.

The register has no employment start/end dates. We assume the supplied register
describes the workforce throughout the export. A historical employee-week with no
shift records is uncertain, not a confirmed zero-hour week.


In [2]:
inventory = pd.DataFrame([
    {"file": name + ".csv", "rows": len(df), "columns": len(df.columns),
     "duplicate_rows": int(df.duplicated().sum()),
     "blank_cells": int(df.apply(lambda col: col.str.strip().eq("").sum()).sum()),
     "fields": ", ".join(df.columns)}
    for name, df in raw.items()
])
display(inventory)

def assert_key(df, columns):
    assert not df[columns].apply(lambda c: c.str.strip().eq("")).any().any()
    assert not df.duplicated(columns).any(), f"Duplicate key: {columns}"

for df, key in [(employees, ["employee_id"]), (sites, ["site_id"]),
                (raw["shifts"], ["shift_id"]), (notes, ["shift_id"]),
                (payroll, ["employee_id"]), (holidays, ["date"]),
                (summary, ["employee_id", "week_starting"])]:
    assert_key(df, key)

employee_ids = set(employees.employee_id)
site_ids = set(sites.site_id)
assert set(raw["shifts"].employee_id) <= employee_ids
assert set(raw["shifts"].site_id) <= site_ids
assert set(employees.primary_site_id) <= site_ids
assert set(notes.shift_id) <= set(raw["shifts"].shift_id)
assert set(summary.employee_id) <= employee_ids
assert set(payroll.employee_id) == employee_ids

employees["contract_ordinary_hours"] = pd.to_numeric(employees.contract_ordinary_hours)
assert employees.contract_ordinary_hours.eq(45).all(), "Review the fixed 55-hour rule."
holidays["date"] = pd.to_datetime(holidays.date, format="%Y-%m-%d", errors="raise")
summary["week_starting"] = pd.to_datetime(summary.week_starting, format="%Y-%m-%d")
assert summary.week_starting.dt.dayofweek.eq(0).all()
for col in ["total_hours", "overtime_hours", "breached"]:
    summary[col] = pd.to_numeric(summary[col], errors="raise")
assert np.isfinite(summary[["total_hours", "overtime_hours", "breached"]]).all().all()
assert summary.total_hours.ge(0).all()
assert summary.breached.isin([0, 1]).all()
print("Empty notes:", notes.note.str.strip().eq("").sum())
print("Literal n/a notes:", notes.note.str.strip().str.lower().eq("n/a").sum())
display(employees.groupby(["role", "shift_pattern"]).size().rename("employees").reset_index())


                  file  rows  columns  duplicate_rows  blank_cells  \
0        employees.csv   213        8               0            0   
1            sites.csv     6        3               0            0   
2           shifts.csv  8863        6               0          184   
3      shift_notes.csv  2117        3               0           22   
4  public_holidays.csv     2        2               0            0   
5   weekly_summary.csv  2122        5               0            0   
6  payroll_details.csv   213       10               0            0   

                                              fields  
0  employee_id, full_name, id_number, role, prima...  
1                       site_id, site_name, province  
2  shift_id, employee_id, site_id, shift_date, cl...  
3                          shift_id, logged_by, note  
4                                         date, name  
5  employee_id, week_starting, total_hours, overt...  
6  employee_id, full_name, id_number, bank_name, ...  

## Parse times and infer the reporting period

Missing clock-outs remain missing. A malformed nonblank time or zero duration is
flagged as invalid. Recorded hours sum only valid completed durations; a total of
zero with missing records does not mean no hours were worked.

The latest `shift_date` supplies the default reporting date, not today's system
date and not the latest clock-out timestamp. We assume that date is fully covered
under the shift-date convention. The export has no completeness metadata; a
future application should display this date for confirmation.


In [3]:
shifts = raw["shifts"].copy()
shifts["shift_date"] = pd.to_datetime(shifts.shift_date, format="%Y-%m-%d", errors="raise")
assert shifts.shift_date.notna().all() and len(shifts) > 0
shifts["week_start"] = shifts.shift_date - pd.to_timedelta(shifts.shift_date.dt.dayofweek, unit="D")

def parse_clock(series):
    text = series.str.strip()
    valid = text.str.fullmatch(r"(?:[01]\d|2[0-3]):[0-5]\d", na=False)
    return pd.to_timedelta(text.where(valid) + ":00", errors="coerce")

clock_in = parse_clock(shifts.clock_in_time)
clock_out = parse_clock(shifts.clock_out_time)
shifts["missing_clockout"] = shifts.clock_out_time.str.strip().eq("")
shifts["start_at"] = shifts.shift_date + clock_in
shifts["end_at"] = shifts.shift_date + clock_out
shifts["overnight"] = (clock_out < clock_in).fillna(False)
shifts.loc[shifts.overnight, "end_at"] += pd.Timedelta(days=1)
duration = (shifts.end_at - shifts.start_at).dt.total_seconds() / 3600
shifts["invalid_time"] = (
    clock_in.isna() | (~shifts.missing_clockout & clock_out.isna())
    | duration.le(0) | duration.ge(24)
)
shifts["recorded_hours"] = duration.mask(shifts.invalid_time)
assert shifts.recorded_hours.dropna().gt(0).all()
assert shifts.recorded_hours.dropna().lt(24).all()
assert shifts.loc[shifts.missing_clockout, "recorded_hours"].isna().all()

REPORTING_DATE = shifts.shift_date.max()
CURRENT_WEEK = REPORTING_DATE - pd.Timedelta(days=REPORTING_DATE.dayofweek)
weeks = pd.date_range(shifts.week_start.min(), CURRENT_WEEK, freq="7D")
coverage = (shifts.groupby("week_start").agg(
    rows=("shift_id", "size"), employees=("employee_id", "nunique"),
    dates_present=("shift_date", "nunique"),
    missing_clockouts=("missing_clockout", "sum"), invalid_times=("invalid_time", "sum"),
    overnight_shifts=("overnight", "sum"),
).reindex(weeks, fill_value=0).rename_axis("week_start").reset_index())
coverage["week_end"] = coverage.week_start + pd.Timedelta(days=6)
# Conservative coverage proxy, not proof that every site's export is complete.
coverage["calendar_coverage_ok"] = coverage.dates_present.eq(7) & coverage.week_end.le(REPORTING_DATE)
print("Shift-date range:", shifts.shift_date.min().date(), "to", REPORTING_DATE.date())
print("Current week:", CURRENT_WEEK.date(), "| reporting day:", REPORTING_DATE.day_name())
print("Latest supplied clock-out:", shifts.end_at.max())
display(coverage)
display(shifts.recorded_hours.describe().to_frame())


Shift-date range: 2026-06-08 to 2026-08-12
Current week: 2026-08-10 | reporting day: Wednesday
Latest supplied clock-out: 2026-08-13 08:15:00
  week_start  rows  employees  dates_present  missing_clockouts  \
0 2026-06-08   921        213              7                 15   
1 2026-06-15   963        213              7                 13   
2 2026-06-22   944        213              7                 25   
3 2026-06-29   915        213              7                 19   
4 2026-07-06   941        213              7                 26   
5 2026-07-13   938        213              7                 19   
6 2026-07-20   942        213              7                 24   
7 2026-07-27   929        213              7                 15   
8 2026-08-03   940        213              7                 25   
9 2026-08-10   430        207              3                  3   

   invalid_times  overnight_shifts   week_end  calendar_coverage_ok  
0              0               103 2026-06-14     

## Detect overlaps without removing records

Two valid intervals overlap when their intersection has positive duration.
Touching endpoints do not overlap. Detection works across sites, midnight and
week boundaries, and records every pair even when several intervals overlap.
Missing/invalid ends cannot be checked reliably; those remain separate flags.

`all_overlap_pairs` is an audit of the entire export only. Snapshot features below
recompute overlaps from the data visible at their own cut-off. Do not merge the
full-export flags into historical features.


In [4]:
def find_overlaps(frame):
    columns = ["employee_id", "shift_id_left", "shift_id_right", "overlap_hours"]
    pairs = []
    valid = frame.loc[frame.recorded_hours.notna()].sort_values(["employee_id", "start_at", "shift_id"])
    for employee_id, group in valid.groupby("employee_id", sort=False):
        active = []
        for row in group.itertuples(index=False):
            active = [previous for previous in active if previous.end_at > row.start_at]
            for previous in active:
                hours = (min(previous.end_at, row.end_at) - row.start_at).total_seconds() / 3600
                pairs.append((employee_id, previous.shift_id, row.shift_id, hours))
            active.append(row)
    return pd.DataFrame(pairs, columns=columns)

all_overlap_pairs = find_overlaps(shifts)
overlap_ids = set(all_overlap_pairs.shift_id_left) | set(all_overlap_pairs.shift_id_right)
print("Overlap pairs:", len(all_overlap_pairs), "| affected shifts:", len(overlap_ids))
display(all_overlap_pairs.head(10))
quality_records = shifts.loc[
    shifts.missing_clockout | shifts.invalid_time | shifts.shift_id.isin(overlap_ids),
    ["shift_id", "employee_id", "site_id", "shift_date", "start_at", "end_at",
     "recorded_hours", "missing_clockout", "invalid_time"]
].copy()
quality_records["overlap_in_full_export"] = quality_records.shift_id.isin(overlap_ids)
display(quality_records.head(10))


Overlap pairs: 126 | affected shifts: 252
  employee_id shift_id_left shift_id_right  overlap_hours
0       E1005       S100208        S100210           4.00
1       E1006       S100248        S100252           4.00
2       E1006       S100253        S100256           4.00
3       E1007       S100295        S100298           4.00
4       E1015       S100630        S100634           4.00
5       E1015       S100635        S100632           3.75
6       E1016       S100668        S100671           4.00
7       E1019       S100818        S100820           4.00
8       E1020       S100850        S100853           4.00
9       E1020       S100851        S100854           4.00
    shift_id employee_id site_id shift_date            start_at  \
96   S107040       E1161   ST-05 2026-06-08 2026-06-08 08:00:00   
97   S107043       E1161   ST-01 2026-06-08 2026-06-08 08:00:00   
138  S100245       E1006   ST-04 2026-06-09 2026-06-09 07:15:00   
154  S101273       E1030   ST-03 2026-06-09 2026-06-

## Build a snapshot from visible records only

This function uses no weekly summary, outcome table, full-export overlap flags,
payroll fields or note classifications. Static register attributes are assumed
valid throughout the period. The holiday calendar is assumed known in advance.

No imputation is performed: `imputed_hours_so_far` is explicitly zero. Missing
shift durations remain unknown. The mean duration is missing if no completed
shift exists. Every registered employee is retained, including those with no
records. A prior Sunday's interval can flag an overlap with a Monday shift, but
its hours are never added to the new week.


In [5]:
REGISTER_COLUMNS = ["employee_id", "role", "primary_site_id", "shift_pattern",
                    "employment_type", "contract_ordinary_hours"]

def build_snapshot(source_shifts, week_start, cutoff_date):
    week_start, cutoff_date = pd.Timestamp(week_start), pd.Timestamp(cutoff_date)
    assert week_start.dayofweek == 0
    assert week_start <= cutoff_date <= week_start + pd.Timedelta(days=6)
    # One previous day suffices for overlap detection under the <24-hour assumption.
    visible = source_shifts.loc[source_shifts.shift_date.between(
        week_start - pd.Timedelta(days=1), cutoff_date)].copy()
    pairs = find_overlaps(visible)
    flagged_ids = set(pairs.shift_id_left) | set(pairs.shift_id_right)
    current = visible.loc[visible.shift_date.ge(week_start)].copy()
    current["overlap_known"] = current.shift_id.isin(flagged_ids)
    current["completed"] = current.recorded_hours.notna()
    assert current.shift_date.le(cutoff_date).all()

    totals = current.groupby("employee_id").agg(
        recorded_hours_so_far=("recorded_hours", "sum"),
        shifts_so_far=("shift_id", "size"),
        completed_shifts_so_far=("completed", "sum"),
        days_worked_so_far=("shift_date", "nunique"),
        mean_recorded_shift_hours_so_far=("recorded_hours", "mean"),
        missing_clockouts_so_far=("missing_clockout", "sum"),
        invalid_time_records_so_far=("invalid_time", "sum"),
        overlapping_shifts_so_far=("overlap_known", "sum"),
        overnight_shifts_so_far=("overnight", "sum"),
        sites_worked_so_far=("site_id", "nunique"),
        latest_shift_date_used=("shift_date", "max"),
    ).reset_index()
    result = employees[REGISTER_COLUMNS].merge(totals, on="employee_id", how="left", validate="one_to_one")
    count_columns = [c for c in totals.columns if c.endswith("_so_far")
                     and c not in ["recorded_hours_so_far", "mean_recorded_shift_hours_so_far"]]
    result[count_columns] = result[count_columns].fillna(0).astype("int64")
    result["recorded_hours_so_far"] = result.recorded_hours_so_far.fillna(0.0)
    result["imputed_hours_so_far"] = 0.0
    result["no_shifts_so_far"] = result.shifts_so_far.eq(0)
    result["week_start"] = week_start
    result["cutoff_date"] = cutoff_date
    result["cutoff_day"] = cutoff_date.dayofweek + 1  # Monday=1
    result["days_remaining"] = 7 - result.cutoff_day
    week_end = week_start + pd.Timedelta(days=6)
    result["holiday_days_elapsed"] = int(holidays.date.between(week_start, cutoff_date).sum())
    result["holiday_days_remaining"] = int(((holidays.date > cutoff_date) & (holidays.date <= week_end)).sum())

    assert len(result) == len(employees) and result.employee_id.is_unique
    assert set(result.employee_id) == employee_ids
    assert result.latest_shift_date_used.dropna().le(cutoff_date).all()
    return result.sort_values("employee_id").reset_index(drop=True)


## Weekly recorded totals, reconciliation and outcome eligibility

Full-week results are held in a separate table. They must never be used as current
week features. The latest represented week is always reserved for current scoring.
Earlier weeks also need seven represented calendar dates, at least one employee
shift, no missing/invalid durations, and no detected overlap affecting their shifts.

This conservative rule is an **evaluation policy**, not proof of physical hours.
Weeks excluded for quality are retained with a reason and a missing target. We do
not promote imputed totals or sub-55 incomplete totals to ground truth. A recorded
breach indicator is retained for auditing but is a usable target only when eligible.

Reconciliation checks the client's arithmetic, not independent truth. Missing
summary rows are displayed separately, including employees with no completed
current shift. They are not silently converted into confirmed zero-hour outcomes.


In [6]:
weekly_parts = []
for week in weeks:
    end = min(week + pd.Timedelta(days=6), REPORTING_DATE)
    part = build_snapshot(shifts, week, end)
    weekly_parts.append(part[["employee_id", "week_start", "recorded_hours_so_far",
                             "shifts_so_far", "missing_clockouts_so_far",
                             "invalid_time_records_so_far", "overlapping_shifts_so_far"]])

weekly_outcomes = pd.concat(weekly_parts, ignore_index=True).rename(columns={
    "recorded_hours_so_far": "outcome_recorded_hours",
    "shifts_so_far": "outcome_shift_count",
    "missing_clockouts_so_far": "outcome_missing_clockouts",
    "invalid_time_records_so_far": "outcome_invalid_times",
    "overlapping_shifts_so_far": "outcome_overlapping_shifts",
}).merge(coverage[["week_start", "calendar_coverage_ok"]], on="week_start", validate="many_to_one")
weekly_outcomes["outcome_recorded_breach"] = weekly_outcomes.outcome_recorded_hours.gt(BREACH_HOURS)

def exclusion_reason(row):
    reasons = []
    if row.week_start == CURRENT_WEEK:
        reasons.append("reserved_current_week")
    if not row.calendar_coverage_ok:
        reasons.append("incomplete_calendar_coverage")
    if row.outcome_shift_count == 0:
        reasons.append("no_shift_records")
    if row.outcome_missing_clockouts > 0:
        reasons.append("missing_clockout")
    if row.outcome_invalid_times > 0:
        reasons.append("invalid_time")
    if row.outcome_overlapping_shifts > 0:
        reasons.append("overlap")
    return "; ".join(reasons)

weekly_outcomes["label_exclusion_reason"] = weekly_outcomes.apply(exclusion_reason, axis=1)
weekly_outcomes["label_eligible"] = weekly_outcomes.label_exclusion_reason.eq("")
weekly_outcomes["target_will_breach"] = weekly_outcomes.outcome_recorded_breach.astype("Int64").where(
    weekly_outcomes.label_eligible)
assert weekly_outcomes.loc[weekly_outcomes.week_start.eq(CURRENT_WEEK), "target_will_breach"].isna().all()

reconciliation = weekly_outcomes.merge(
    summary.rename(columns={"week_starting": "week_start"}),
    on=["employee_id", "week_start"], how="outer", validate="one_to_one", indicator=True)
reconciliation["hours_difference"] = reconciliation.outcome_recorded_hours - reconciliation.total_hours
matched = reconciliation._merge.eq("both")
display(reconciliation.groupby("_merge", observed=True).agg(
    employee_weeks=("employee_id", "size"),
    max_absolute_difference=("hours_difference", lambda x: x.abs().max())))
display(reconciliation.loc[~matched, ["employee_id", "week_start", "outcome_shift_count",
                                    "outcome_missing_clockouts", "_merge"]])
assert not reconciliation._merge.eq("right_only").any(), "Summary contains unsupported weeks."
assert np.allclose(reconciliation.loc[matched, "hours_difference"], 0, atol=1e-8, rtol=0)
assert np.allclose(summary.overtime_hours, (summary.total_hours - 45).clip(lower=0), atol=1e-8, rtol=0)
assert summary.breached.eq(summary.total_hours.gt(BREACH_HOURS).astype(int)).all()
assert not weekly_outcomes.duplicated(["employee_id", "week_start"]).any()


           employee_weeks  max_absolute_difference
_merge                                            
left_only               8                     <NA>
both                 2122                      0.0
     employee_id week_start  outcome_shift_count  outcome_missing_clockouts  \
89         E1009 2026-08-10                    0                          0   
939        E1094 2026-08-10                    1                          1   
1009       E1101 2026-08-10                    0                          0   
1449       E1145 2026-08-10                    0                          0   
1569       E1157 2026-08-10                    0                          0   
1699       E1170 2026-08-10                    0                          0   
1819       E1182 2026-08-10                    1                          1   
1959       E1196 2026-08-10                    0                          0   

         _merge  
89    left_only  
939   left_only  
1009  left_only  
1449  left_o

## Historical partial weeks and the current snapshot

Historical features are built first. Eventual outcomes are attached afterward,
with outcome and eligibility fields clearly separated from candidate features.
The eligible historical table is a benchmark subset, not the whole operational
population. Do not describe its performance as performance on missing/overlapping
records without a separate evaluation.

All historical weeks are retained, including early weeks that a later model may
need only as a training/history warm-up. No train/test split or historical
imputation is fitted in this stage. Wednesday is configurable, while the current
snapshot always uses the inferred reporting date.


In [7]:
historical_weeks = [week for week in weeks if week < CURRENT_WEEK]
assert historical_weeks, "No earlier weeks available for backtesting."
historical_features = pd.concat([
    build_snapshot(shifts, week, week + pd.Timedelta(days=HISTORICAL_CUTOFF_DAY))
    for week in historical_weeks
], ignore_index=True)

historical_snapshot = historical_features.merge(
    weekly_outcomes, on=["employee_id", "week_start"], how="left", validate="one_to_one")
historical_usable = historical_snapshot.loc[historical_snapshot.label_eligible].copy()
current_snapshot = build_snapshot(shifts, CURRENT_WEEK, REPORTING_DATE)

# Explicit allow-list. Do not derive X by dropping only the target column.
FEATURE_COLUMNS = [
    "recorded_hours_so_far", "shifts_so_far", "completed_shifts_so_far",
    "days_worked_so_far", "mean_recorded_shift_hours_so_far",
    "missing_clockouts_so_far", "invalid_time_records_so_far",
    "overlapping_shifts_so_far", "overnight_shifts_so_far", "sites_worked_so_far",
    "no_shifts_so_far", "cutoff_day", "days_remaining",
    "holiday_days_elapsed", "holiday_days_remaining", "role", "shift_pattern",
]
X_historical = historical_usable[FEATURE_COLUMNS].copy()
y_historical = historical_usable.target_will_breach.astype("int64").copy()
X_current = current_snapshot[FEATURE_COLUMNS].copy()
# Categorical encoding, missing-mean handling, feature selection and fitting are deferred.

assert not historical_snapshot.duplicated(["employee_id", "week_start"]).any()
assert historical_snapshot.groupby("week_start").size().eq(len(employees)).all()
assert historical_snapshot.week_start.lt(CURRENT_WEEK).all()
assert historical_snapshot.cutoff_date.dt.dayofweek.eq(HISTORICAL_CUTOFF_DAY).all()
assert historical_snapshot.target_will_breach.notna().eq(historical_snapshot.label_eligible).all()
assert len(current_snapshot) == len(employees) and current_snapshot.employee_id.is_unique
assert set(current_snapshot.employee_id) == employee_ids
assert not any(c.startswith("outcome_") or c.startswith("label_") or c == "target_will_breach"
               for c in FEATURE_COLUMNS)
assert not set(weekly_outcomes.columns[2:]) & set(current_snapshot.columns)

# Prefix invariance: removing all future records must leave each snapshot unchanged.
for week in historical_weeks:
    cutoff = week + pd.Timedelta(days=HISTORICAL_CUTOFF_DAY)
    from_full_export = build_snapshot(shifts, week, cutoff)
    from_truncated_export = build_snapshot(shifts.loc[shifts.shift_date.le(cutoff)].copy(), week, cutoff)
    pd.testing.assert_frame_equal(from_full_export, from_truncated_export)
print("Snapshot coverage, target separation and prefix-invariance checks passed.")


Snapshot coverage, target separation and prefix-invariance checks passed.


## Review counts, schemas and uncertain cases

`historical_snapshot` retains every historical employee-week. `historical_usable`
has the same schema but contains only eligible labels. `current_snapshot` has no
outcome or label fields. The explicit feature allow-list is provisional: it does
not imply that every candidate should enter a later model.

Retention counts below describe progressively stricter filters. Breaches in the
early rows are recorded arithmetic flags; they become eligible targets only in
the final row. Invalid/missing/overlapping records may have unknown true outcomes.


In [8]:
h = historical_snapshot
masks = [
    ("All historical employee-weeks", pd.Series(True, index=h.index)),
    ("Complete calendar coverage and at least one shift", h.calendar_coverage_ok & h.outcome_shift_count.gt(0)),
]
masks.append(("Also no missing clock-outs", masks[-1][1] & h.outcome_missing_clockouts.eq(0)))
masks.append(("Also no invalid times", masks[-1][1] & h.outcome_invalid_times.eq(0)))
masks.append(("Also no overlaps: eligible ground truth", masks[-1][1] & h.outcome_overlapping_shifts.eq(0)))
retention = pd.DataFrame([
    {"stage": name, "employee_weeks": int(mask.sum()),
     "recorded_breaches": int(h.loc[mask, "outcome_recorded_breach"].sum())}
    for name, mask in masks
])
assert masks[-1][1].eq(h.label_eligible).all()
display(retention)
display(h.groupby("week_start").agg(
    employee_weeks=("employee_id", "size"), eligible=("label_eligible", "sum"),
    eligible_breaches=("target_will_breach", lambda x: x.sum(min_count=1))))
display(h.loc[~h.label_eligible, "label_exclusion_reason"].value_counts().rename("employee_weeks").to_frame())

def schema_table(frame):
    def purpose(column):
        if column in FEATURE_COLUMNS:
            return "candidate feature"
        if column.startswith(("outcome_", "label_")) or column in ["target_will_breach", "calendar_coverage_ok"]:
            return "OUTCOME / evaluation only"
        return "identifier / context / audit"
    return pd.DataFrame({"column": frame.columns,
                         "dtype": [str(frame[c].dtype) for c in frame],
                         "nulls": [int(frame[c].isna().sum()) for c in frame],
                         "purpose": [purpose(c) for c in frame]})

print("HISTORICAL SCHEMA (same columns for historical_usable)")
display(schema_table(historical_snapshot))
print("CURRENT SCHEMA")
display(schema_table(current_snapshot))
print("Usable historical employee-weeks:", len(historical_usable))
print("Usable recorded breaches:", int(y_historical.sum()))
print("Current employees:", len(current_snapshot))
display(pd.Series({
    "no_current_shift_records": int(current_snapshot.no_shifts_so_far.sum()),
    "employees_with_missing_clockouts": int(current_snapshot.missing_clockouts_so_far.gt(0).sum()),
    "employees_with_no_completed_shifts": int(current_snapshot.completed_shifts_so_far.eq(0).sum()),
    "employees_with_overlaps": int(current_snapshot.overlapping_shifts_so_far.gt(0).sum()),
    "missing_clockout_records": int(current_snapshot.missing_clockouts_so_far.sum()),
}, name="current_week_count").to_frame())
display(current_snapshot.loc[
    current_snapshot.no_shifts_so_far | current_snapshot.missing_clockouts_so_far.gt(0)
    | current_snapshot.overlapping_shifts_so_far.gt(0),
    ["employee_id", "recorded_hours_so_far", "shifts_so_far", "completed_shifts_so_far",
     "missing_clockouts_so_far", "overlapping_shifts_so_far", "no_shifts_so_far"]])


                                               stage  employee_weeks  \
0                      All historical employee-weeks            1917   
1  Complete calendar coverage and at least one shift            1917   
2                         Also no missing clock-outs            1747   
3                              Also no invalid times            1747   
4            Also no overlaps: eligible ground truth            1666   

   recorded_breaches  
0                 66  
1                 66  
2                 66  
3                 66  
4                 61  
            employee_weeks  eligible  eligible_breaches
week_start                                             
2026-06-08             213       191                  6
2026-06-15             213       188                  4
2026-06-22             213       180                  7
2026-06-29             213       186                  7
2026-07-06             213       180                  7
2026-07-13             213       192 

## Assumptions to resolve before comparing predictors

- Confirm that hidden evaluation uses the shift-start-date convention rather than
  splitting overnight hours at calendar-week boundaries. This notebook deliberately
  follows the supplied summary's arithmetic; it is not an independent legal-hours audit.
- Confirm how simultaneous records at different sites should be interpreted. All
  records are preserved; affected employee-weeks are excluded from clean labels.
- Export completeness and historical record availability are assumptions. Seven
  represented dates do not prove that every site or employee was fully reported.
- No employment-history dates, breaks, roster or planned remaining shifts are
  supplied. No-record weeks and missing clock-outs cannot be confirmed as zero work.
- No hours are imputed. Before predicting uncertain current cases, choose a
  past-only duration estimate or explicit sensitivity scenarios. Never use an
  imputed Sunday total as a verified training label.
- The clean benchmark excludes hard cases, including overlapping/missing records.
  All 213 current employees still need a prediction in a later stage.
- The current register is assumed valid for earlier weeks. Notes lack creation
  timestamps and are not used as features in this stage.
- The holiday calendar contains only the supplied dates. Future exports must
  include holidays still to come in the reporting week if those features are used.
- Wednesday overnight completions can be used only under the documented reporting
  convention. A literal midnight deployment needs separate in-progress-shift logic.

Next stage, after review: choose chronological development/test weeks, fit all
history-derived transformations inside each training window, and compare a small
set of methods. Do not use `outcome_*`, `label_*`, `calendar_coverage_ok`, or
`target_will_breach` as predictors. No prediction method has been built here.
